# CS-781 Phase 5 - final ensemble (CPU is enough)

Combines the cached dev/test scores of the full-data runs (nothing is trained; CPU, a few minutes). Each member's logits are shifted per statute so that its own best threshold sits at 0 (members are calibrated differently), the shifted logits are averaged, and the thresholds (global / per-statute / shrunk) are fitted again on the ensemble. Members are aligned by case id on the selection set (dev cases whose text is not in test). Ensembles: greedy selection, top-3, top-5, all members within 3% of the best, best single. An honest check chooses members and thresholds on one half of the selection set and scores the other half: use it to decide what to submit. Writes one submission zip per ensemble. Missing members are skipped.

In [1]:
TOKENIZER_NAME = None
!pip install -q -U "huggingface_hub>=1.5"
import os, json, shutil, subprocess, sys, time, zipfile, random, math, gc
from collections import Counter, defaultdict
import torch, torch.nn as nn, torch.nn.functional as F
from huggingface_hub import hf_hub_download, login, download_bucket_files, batch_bucket_files
from kaggle_secrets import UserSecretsClient

NB_START = time.time()
WORK = "/kaggle/working"
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_BUCKET = "kavincg23/cs781-models"
HF_CHECKPOINT_REPO = "kavincg23/cs781-checkpoints"   # old repo, read-only fallback
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
HF_WRITE_TOKEN = user_secrets.get_secret("HF_WRITE")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")


def get_dataset_file(filename):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename, repo_type="dataset", token=HF_TOKEN)


def fetch_model_file(name, local_path=None):
    local_path = local_path or f"{WORK}/{name}"
    if os.path.exists(local_path):
        return local_path
    part = local_path + ".part"
    try:
        download_bucket_files(HF_BUCKET, files=[(name, part)], raise_on_missing_files=True, token=HF_WRITE_TOKEN)
        os.replace(part, local_path)
        return local_path
    except Exception:
        if os.path.exists(part):
            os.remove(part)
    # fallback: the OLD model repo (the Phase-1 notebooks do the same; the tokenizer only lives there)
    p = hf_hub_download(repo_id=HF_CHECKPOINT_REPO, filename=name, repo_type="model", token=HF_WRITE_TOKEN)
    shutil.copy(p, local_path)
    print(f"(found {name} in the old repo {HF_CHECKPOINT_REPO})")
    return local_path


def upload_model_file(local_path, name=None):
    name = name or os.path.basename(local_path)
    try:
        batch_bucket_files(HF_BUCKET, add=[(local_path, name)], token=HF_WRITE_TOKEN)
        print(f"uploaded {name} to bucket {HF_BUCKET}")
    except Exception as e:
        print(f"(upload of {name} failed: {type(e).__name__}: {e}; the file is still in {WORK})")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 855.2/855.2 kB 6.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.3.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.3.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.3.0 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 2.3.0 which is incompatible.
gradio-client 2.6.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.3.0 which is incompatible.
tokenizers 0.23.1 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.3.0 which is incompatible.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


device: cpu 


In [2]:
# ===================== Phase 5 (Legal Statute Identification) engine: long-document chunking, chunk pooling, multi-label document heads, losses, thresholds, training, inference =====================
# Works with any Phase-1 model wrapped as a "backbone" exposing  features(ids, mask, soft_prompt) -> (B, T, d_model).
# A document = its BPE token ids. It is cut into CHUNKS of <= WINDOW tokens (sliding window, stride STRIDE); at most MAX_CHUNKS chunks are kept
# (SELECT says which). The backbone reads every chunk as one row, a chunk is pooled to one vector (mean of its tokens and/or its last token), a document
# head (mean / attention / BiLSTM + attention / one attention per statute) turns the chunk vectors into L statute scores (logits, one per statute id).
import math, time, random, json, os, copy, zipfile, hashlib
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


# ----------------------------------------------------------------------------------------------------------------------
# 1. Documents, chunks
# ----------------------------------------------------------------------------------------------------------------------
def doc_text(r):
    t = r["text"]
    return " ".join(t) if isinstance(t, list) else str(t)


def text_hash(t):
    return hashlib.md5((t[:2000] + "||" + t[-2000:]).encode("utf-8", errors="ignore")).hexdigest()


def label_list(v):
    return sorted(set(int(x) for x in (v if isinstance(v, (list, tuple)) else [v])))


def multi_hot(labels, L):
    y = np.zeros(L, dtype=np.uint8)
    if len(labels):
        y[list(labels)] = 1
    return y


def encode_rows(tok, rows, id_key, label_key, max_tokens, log=print):
    """-> list of dict(id, ids=np.uint16 array, labs=list|None, n). Documents longer than max_tokens keep their first and last max_tokens//2 tokens."""
    out, t0, cut = [], time.time(), 0
    for i, r in enumerate(rows):
        ids = tok.encode(doc_text(r))
        if not ids:
            ids = [1]
        if len(ids) > max_tokens:
            ids = ids[:max_tokens // 2] + ids[-(max_tokens - max_tokens // 2):]; cut += 1
        out.append(dict(id=r[id_key], ids=np.asarray(ids, dtype=np.uint16), labs=label_list(r[label_key]) if label_key and r.get(label_key) is not None else None, n=len(ids)))
        if (i + 1) % 5000 == 0:
            log(f"    tokenised {i + 1}/{len(rows)} documents [{time.time() - t0:.0f}s]")
    return out, dict(documents=len(out), tokens=int(sum(d["n"] for d in out)), cut=cut)


def cue_positions(ids, patterns):
    """Start positions of the cue token patterns (the BPE tokens of <SECTION> / <ACT>: where the masked statute mentions were) in a token array."""
    a = np.asarray(ids); out = []
    for p in patterns:
        k = len(p)
        if k == 0 or len(a) < k:
            continue
        m = np.ones(len(a) - k + 1, dtype=bool)
        for j, t in enumerate(p):
            m &= a[j:len(a) - k + 1 + j] == t
        out.append(np.flatnonzero(m))
    return np.unique(np.concatenate(out)) if out else np.zeros(0, dtype=np.int64)


def make_spans(n, C, d=None):
    """Chunk spans (start, end) of a document with n tokens, after applying MAX_CHUNKS / SELECT.
    SELECT='cue' (evidence-based selection): keep the MAX_CHUNKS chunks with the most cue tokens (<SECTION> / <ACT> mentions); ties -> head+tail order."""
    W, S = C["WINDOW"], max(1, C["STRIDE"])
    if n <= W:
        return [(0, n)]
    starts = list(range(0, n - W + 1, S))
    if starts[-1] + W < n:
        starts.append(n - W)                                   # the last chunk is aligned to the end of the document
    spans = [(s, s + W) for s in starts]
    K, m, how = len(spans), C["MAX_CHUNKS"], C["SELECT"]
    if how == "all" or m is None or K <= m:
        return spans
    if how == "first":
        idx = list(range(m))
    elif how == "last":
        idx = list(range(K - m, K))
    elif how == "middle":
        a = (K - m) // 2; idx = list(range(a, a + m))
    elif how == "head_tail":
        h = min(C["HEAD_CHUNKS"], m - 1) if m > 1 else 0
        idx = list(range(h)) + list(range(K - (m - h), K))
    elif how == "uniform":
        idx = sorted(set(int(round(x)) for x in np.linspace(0, K - 1, m)))
    elif how == "cue":
        if d is not None and "cue" not in d:
            d["cue"] = cue_positions(d["ids"], C["CUE_PATTERNS"])
        cue = d["cue"] if d is not None else np.zeros(0)
        h = min(C["HEAD_CHUNKS"], m - 1) if m > 1 else 0
        ht = set(list(range(h)) + list(range(K - (m - h), K)))
        score = [int(((cue >= s) & (cue < e)).sum()) for s, e in spans]
        idx = sorted(sorted(range(K), key=lambda i: (-score[i], 0 if i in ht else 1, i))[:m])
    else:
        raise ValueError(how)
    return [spans[i] for i in idx]


def collate(docs, idxs, C, with_labels=True, pad_id=0):
    rows, counts, spans_all = [], [], []
    for di in idxs:
        d = docs[di]; sp = make_spans(d["n"], C, d)
        for s, e in sp:
            rows.append(d["ids"][s:e].tolist())
        counts.append(len(sp)); spans_all.append(sp)
    R, T = len(rows), max(len(r) for r in rows)
    ids = torch.full((R, T), pad_id, dtype=torch.long)
    mask = torch.zeros((R, T), dtype=torch.bool)
    for r, a in enumerate(rows):
        ids[r, :len(a)] = torch.tensor(a); mask[r, :len(a)] = True
    K = max(counts)
    chunk_mask = torch.arange(K).unsqueeze(0) < torch.tensor(counts).unsqueeze(1)
    b = dict(ids=ids, mask=mask, counts=counts, chunk_mask=chunk_mask, spans=spans_all)
    if with_labels and all(docs[di].get("y") is not None for di in idxs):
        b["labels"] = torch.from_numpy(np.stack([docs[di]["y"] for di in idxs])).float()
    if with_labels and all(docs[di].get("kd") is not None for di in idxs):
        b["kd"] = torch.stack([torch.as_tensor(docs[di]["kd"]).float() for di in idxs])
    return b


def to_device(b, device):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in b.items()}


# ----------------------------------------------------------------------------------------------------------------------
# 2. Model = backbone + chunk pooling + multi-label document head
# ----------------------------------------------------------------------------------------------------------------------
class DocHead(nn.Module):
    """chunk vectors (B, K, d_in) + chunk mask -> statute logits (B, L).
    kind: 'mean' | 'attn' (one attention pooling shared by all statutes) | 'bilstm' (BiLSTM over the chunk sequence, then shared attention pooling)
          | 'label_attn' (every statute has its own attention query over the chunks: it can look at the chunk that is about it).
    label_mix: a zero-initialised L x L layer on the sigmoid of the first-stage scores lets the head learn label co-occurrence (e.g. statute 12 -> 13)."""

    def __init__(self, d_in, n_labels, kind="bilstm", hidden=256, dropout=0.1, norm=True, label_mix=False):
        super().__init__()
        self.kind, self.L = kind, n_labels
        self.norm = nn.LayerNorm(d_in) if norm else nn.Identity()
        self.drop = nn.Dropout(dropout)
        d = d_in
        self.lstm = None
        if kind == "bilstm":
            self.lstm = nn.LSTM(d_in, hidden, batch_first=True, bidirectional=True)
            d = 2 * hidden
        if kind in ("attn", "bilstm"):
            self.score = nn.Sequential(nn.Linear(d, 128), nn.Tanh(), nn.Linear(128, 1))
            self.out = nn.Linear(d, n_labels)
        elif kind == "label_attn":
            self.key = nn.Sequential(nn.Linear(d, 128), nn.Tanh())
            self.lq = nn.Parameter(torch.randn(n_labels, 128) * 0.1)
            self.W = nn.Parameter(torch.randn(n_labels, d) * (1.0 / math.sqrt(d)))
            self.b = nn.Parameter(torch.zeros(n_labels))
        elif kind in ("mean", "chunk_max", "chunk_mean"):     # chunk_*: statute scores for EVERY chunk, then max / mean over the chunks
            self.out = nn.Linear(d, n_labels)
        else:
            raise ValueError(kind)
        self.mix = nn.Linear(n_labels, n_labels, bias=False) if label_mix else None
        if self.mix is not None:
            nn.init.zeros_(self.mix.weight)
        self.last_attn = None

    @torch.no_grad()
    def init_bias(self, prior):
        """Output bias = logit of the label frequency: training starts from 'every statute at its base rate', not from p = 0.5 for all 100 statutes."""
        b = torch.logit(torch.as_tensor(prior, dtype=torch.float32).clamp(1e-4, 1 - 1e-4))
        (self.b if self.kind == "label_attn" else self.out.bias).copy_(b)

    def forward(self, v, mask):
        x = self.drop(self.norm(v.float()))
        if self.lstm is not None:
            lengths = mask.sum(1).clamp(min=1).cpu()
            packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
            with torch.autocast(device_type=x.device.type, enabled=False):
                o, _ = self.lstm(packed)
            x, _ = nn.utils.rnn.pad_packed_sequence(o, batch_first=True, total_length=v.shape[1])
        if self.kind in ("chunk_max", "chunk_mean"):
            cl = self.out(x)                                                        # (B, K, L) chunk-level statute scores
            if self.kind == "chunk_max":
                cl = cl.masked_fill(~mask.unsqueeze(-1), float("-inf")); logits, am = cl.max(1)
                self.last_attn = torch.stack([(am == k).float().mean(-1) for k in range(cl.shape[1])], 1).detach()   # share of statutes whose max is in chunk k
            else:
                w = mask.float() / mask.float().sum(1, keepdim=True).clamp(min=1); self.last_attn = w.detach()
                logits = (cl * w.unsqueeze(-1)).sum(1)
        elif self.kind == "label_attn":
            s = torch.einsum("bkh,lh->bkl", self.key(x), self.lq).masked_fill(~mask.unsqueeze(-1), float("-inf"))
            w = torch.softmax(s, dim=1)                                            # (B, K, L): each statute's attention over the chunks
            self.last_attn = w.mean(-1).detach()
            logits = (w * torch.einsum("bkd,ld->bkl", x, self.W)).sum(1) + self.b
        else:
            if self.kind == "mean":
                w = mask.float() / mask.float().sum(1, keepdim=True).clamp(min=1)
            else:
                s = self.score(x).squeeze(-1).masked_fill(~mask, float("-inf"))
                w = torch.softmax(s, dim=1)
            self.last_attn = w.detach()
            logits = self.out((x * w.unsqueeze(-1)).sum(1))
        if self.mix is not None:
            logits = logits + self.mix(torch.sigmoid(logits))
        return logits


class DocModel(nn.Module):
    """Attribute names shared with the Phase-3/4 model (backbone, soft_prompt, head_parameters) so the technique code works unchanged."""

    def __init__(self, backbone, d_model, C):
        super().__init__()
        self.backbone = backbone
        self.soft_prompt = None
        self.pool = C["POOL"]
        self.row_group = C["ROW_GROUP"]
        self.ckpt = C.get("CKPT", False)
        if self.ckpt and any(getattr(m, "bias_rate", 0) > 0 or (hasattr(m, "tau") and getattr(type(m)._route, "__name__", "") != "_mor_route_frozen") for m in backbone.modules()):
            print("note: the backbone updates routing state inside its forward pass (MoE bias / MoR threshold): activation checkpointing switched off")
            self.ckpt = False
        d_pool = d_model * (2 if self.pool == "mean+last" else 1)
        self.head = DocHead(d_pool, C["N_LABELS"], C["DOC_HEAD"], C["HEAD_LSTM_HIDDEN"], C["HEAD_DROPOUT"], C["HEAD_NORM"], C.get("LABEL_MIX", False))

    def head_parameters(self):
        return list(self.head.parameters())

    def _pool_rows(self, ids, mask):
        h = self.backbone.features(ids, mask, self.soft_prompt).float()
        parts = []
        if self.pool in ("mean", "mean+last"):
            m = mask.unsqueeze(-1).float()
            parts.append((h * m).sum(1) / m.sum(1).clamp(min=1))
        if self.pool in ("last", "mean+last"):
            last = mask.sum(1) - 1
            parts.append(h[torch.arange(h.shape[0], device=h.device), last])
        return torch.cat(parts, -1)

    def chunk_vectors(self, b):
        ids, mask = b["ids"], b["mask"]
        trainable = self.training and any(p.requires_grad for p in self.backbone.parameters()) or (self.training and self.soft_prompt is not None)
        out = []
        for a in range(0, ids.shape[0], self.row_group):
            i, m = ids[a:a + self.row_group], mask[a:a + self.row_group]
            L = int(m.sum(1).max()); i, m = i[:, :L], m[:, :L]
            if not trainable:
                with torch.no_grad():
                    out.append(self._pool_rows(i, m))
            elif self.ckpt:
                out.append(checkpoint(self._pool_rows, i, m, use_reentrant=False))
            else:
                out.append(self._pool_rows(i, m))
        return torch.cat(out, 0)                                              # (R, d_pool), documents in order

    def forward(self, b):
        v = self.chunk_vectors(b)
        cm = b["chunk_mask"]
        vp = v.new_zeros(cm.shape[0], cm.shape[1], v.shape[1])
        vp[cm] = v                                                            # row-major order of True entries == document order
        return self.head(vp, cm)                                              # (B, L)


# ----------------------------------------------------------------------------------------------------------------------
# 3. Scores and thresholds (macro-F1 over the L statutes is the primary metric; micro-F1 and subset accuracy are secondary)
# ----------------------------------------------------------------------------------------------------------------------
def ml_scores(Y, pred):
    """Y, pred: (N, L) 0/1 arrays. Per-statute F1 is 0 when a statute has no true and no predicted positive (never the case here: every statute has dev support)."""
    Y, pred = np.asarray(Y).astype(bool), np.asarray(pred).astype(bool)
    tp = (Y & pred).sum(0).astype(np.float64); fp = (~Y & pred).sum(0); fn = (Y & ~pred).sum(0)
    p = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0); r = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
    f = np.divide(2 * p * r, p + r, out=np.zeros_like(tp), where=(p + r) > 0)
    TP, FP, FN = tp.sum(), fp.sum(), fn.sum()
    mp, mr = TP / max(TP + FP, 1), TP / max(TP + FN, 1)
    inter = (Y & pred).sum(1).astype(np.float64); den = Y.sum(1) + pred.sum(1)
    sample_f = np.divide(2 * inter, den, out=np.zeros_like(inter), where=den > 0)
    return dict(macro_F1=float(f.mean()), micro_F1=float(2 * mp * mr / max(mp + mr, 1e-12)), subset_acc=float((Y == pred).all(1).mean()), macro_P=float(p.mean()), macro_R=float(r.mean()),
                micro_P=float(mp), micro_R=float(mr), samples_F1=float(sample_f.mean()), hamming=float((Y != pred).mean()), mean_pred_labels=float(pred.sum(1).mean()),
                mean_true_labels=float(Y.sum(1).mean()), per_label_F1=f.tolist(), n=int(len(Y)))


def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-np.clip(np.asarray(x, dtype=np.float64), -30, 30)))


GRID = np.round(np.arange(0.05, 0.951, 0.01), 2)


def apply_thr(P, thr, min_one=True):
    """P (N, L) probabilities, thr scalar or (L,). Every case has at least one statute in the data: if none passes its threshold, the statute with the largest margin P - thr is predicted."""
    thr = np.broadcast_to(np.asarray(thr, dtype=np.float64), (P.shape[1],))
    pred = P > thr
    if min_one:
        none = ~pred.any(1)
        if none.any():
            am = (P[none] - thr).argmax(1); sub = np.zeros((none.sum(), P.shape[1]), dtype=bool); sub[np.arange(len(am)), am] = True; pred[none] = sub
    return pred


def f1_grid(Y, P, grid=GRID):
    """-> F (T, L): per-statute F1 at every global threshold of the grid."""
    Yb = np.asarray(Y).astype(bool); F_ = np.zeros((len(grid), Yb.shape[1]))
    ny = Yb.sum(0).astype(np.float64)
    for k, t in enumerate(grid):
        pr = P > t
        tp = (Yb & pr).sum(0); npred = pr.sum(0)
        F_[k] = np.divide(2.0 * tp, ny + npred, out=np.zeros(Yb.shape[1]), where=(ny + npred) > 0)
    return F_


def fit_thresholds(Y, P, mode="global", shrink=0.5, grid=GRID, smooth=5):
    """-> (L,) thresholds. global: one threshold for all statutes (the one maximising macro-F1); per_label: each statute its own best threshold (on a
    smoothed F1 curve, to avoid chasing noise); shrunk: per_label pulled towards the global threshold (shrink = weight of the per-label value)."""
    F_ = f1_grid(Y, P, grid); L = F_.shape[1]
    tg = float(grid[int(np.argmax(F_.mean(1)))])
    if mode == "global":
        return np.full(L, tg)
    k = max(1, smooth)
    Fs = np.apply_along_axis(lambda c: np.convolve(np.pad(c, (k // 2, k // 2), mode="edge"), np.ones(k) / k, mode="valid"), 0, F_) if k > 1 else F_
    tl = grid[np.argmax(Fs, 0)]
    return tl if mode == "per_label" else shrink * tl + (1 - shrink) * tg


THR_MODES = ("global", "per_label", "shrunk")


def cross_fit(Y, P, modes=THR_MODES, reps=6, min_one=True, seed=0):
    """Honest threshold evaluation: thresholds are fitted on one random half of the cases and scored on the other half. -> {mode: mean macro-F1}, {mode: sd}."""
    res = {m: [] for m in modes}
    for r in range(reps):
        perm = np.random.RandomState(seed + r).permutation(len(Y)); A, B = perm[:len(Y) // 2], perm[len(Y) // 2:]
        for m in modes:
            for tr, te in ((A, B), (B, A)):
                res[m].append(ml_scores(Y[te], apply_thr(P[te], fit_thresholds(Y[tr], P[tr], m), min_one))["macro_F1"])
    return {m: float(np.mean(v)) for m, v in res.items()}, {m: float(np.std(v)) for m, v in res.items()}


def best_global(Y, P, min_one=True):
    t = float(fit_thresholds(Y, P, "global")[0])
    return t, ml_scores(Y, apply_thr(P, t, min_one))["macro_F1"]


# ----------------------------------------------------------------------------------------------------------------------
# 4. Losses
# ----------------------------------------------------------------------------------------------------------------------
def pos_weights(Ytrain, power, cap):
    """pos_weight_l = ((N - n_l) / n_l) ** power, capped (power 0.5 = inverse square root of the odds)."""
    n = Ytrain.sum(0).astype(np.float64); N = len(Ytrain)
    w = np.minimum(((N - n) / np.maximum(n, 1)) ** power, cap)
    return torch.tensor(w / w.mean(), dtype=torch.float32)


def ml_loss(logits, y, C, pw=None):
    """LOSS: bce | bce_pw (pos_weight per statute) | focal | asl (asymmetric loss, Ridnik et al. 2021). Returns the mean over the batch of the per-statute mean x LOSS_SCALE."""
    eps = C.get("LABEL_SMOOTHING", 0.0)
    if eps:
        y = y * (1 - eps) + eps / 2
    kind = C["LOSS"]
    if kind in ("bce", "bce_pw"):
        l = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pw if kind == "bce_pw" else None, reduction="none")
    elif kind == "focal":
        bce = F.binary_cross_entropy_with_logits(logits, y, reduction="none"); p = torch.sigmoid(logits)
        pt = y * p + (1 - y) * (1 - p)
        l = bce * (1 - pt) ** C["FOCAL_GAMMA"]
    elif kind == "asl":
        xp = torch.sigmoid(logits); xn = 1 - xp
        if C["ASL_CLIP"] > 0:
            xn = (xn + C["ASL_CLIP"]).clamp(max=1)
        lp = y * torch.log(xp.clamp(min=1e-8)); ln = (1 - y) * torch.log(xn.clamp(min=1e-8))
        pt = xp * y + xn * (1 - y)
        g = C["ASL_GAMMA_POS"] * y + C["ASL_GAMMA_NEG"] * (1 - y)
        l = -(lp + ln) * (1 - pt).detach().pow(g)
    else:
        raise ValueError(kind)
    return l.mean(1).mean() * C["LOSS_SCALE"]


def kd_loss(logits, teacher_logits, T, C):
    """Per-statute distillation: the student's tempered sigmoid is pulled to the teacher's tempered sigmoid (binary cross-entropy with soft targets, x T^2)."""
    soft = torch.sigmoid(teacher_logits / T)
    return F.binary_cross_entropy_with_logits(logits / T, soft, reduction="none").mean(1).mean() * T * T * C["LOSS_SCALE"]


# ----------------------------------------------------------------------------------------------------------------------
# 5. Training
# ----------------------------------------------------------------------------------------------------------------------
def _lr_lambda(total, warmup):
    def f(step):
        if step < warmup:
            return (step + 1) / max(1, warmup)
        return max(0.0, (total - step) / max(1, total - warmup))
    return f


def trainable_state(model):
    st = {n: p.detach().to("cpu", copy=True) for n, p in model.named_parameters() if p.requires_grad}
    for n, bf in model.named_buffers():
        if n.endswith(("select_bias", ".tau", ".tau_ready", ".ada_mask")):
            st[n] = bf.detach().to("cpu", copy=True)
    return st


def load_trainable_state(model, st):
    known = dict(model.named_parameters()); known.update(dict(model.named_buffers()))
    missing = [k for k in st if k not in known]
    assert not missing, f"state has unknown keys: {missing[:5]}"
    model.load_state_dict(st, strict=False)


@torch.no_grad()
def predict_logits(model, docs, C, device, collect_attn=False):
    """-> (N, L) float32 logits in the order of docs, plus timing; with collect_attn also the per-document chunk attention weights and spans."""
    model.eval(); model.to(device)
    order = sorted(range(len(docs)), key=lambda i: len(make_spans(docs[i]["n"], C, docs[i])))        # similar chunk counts together: less padding
    out = np.zeros((len(docs), C["N_LABELS"]), dtype=np.float32)
    attn = {} if collect_attn else None
    use_amp = device.type == "cuda" and C["AMP"] == "fp16"
    t0, tokens = time.time(), 0
    for a in range(0, len(order), C["INFER_BATCH"]):
        idx = order[a:a + C["INFER_BATCH"]]
        b = to_device(collate(docs, idx, C, with_labels=False), device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            lg = model(b)
        out[idx] = lg.float().cpu().numpy(); tokens += int(b["mask"].sum())
        if collect_attn:
            w = model.head.last_attn.cpu().numpy()
            for j, di in enumerate(idx):
                attn[docs[di]["id"]] = dict(w=w[j, :b["counts"][j]].tolist(), spans=b["spans"][j], n=docs[di]["n"])
    if device.type == "cuda":
        torch.cuda.synchronize()
    secs = time.time() - t0
    info = dict(seconds=round(secs, 2), tokens=tokens, tokens_per_sec=round(tokens / max(secs, 1e-9), 1), docs_per_sec=round(len(docs) / max(secs, 1e-9), 2))
    return (out, info, attn) if collect_attn else (out, info)


def train_doc(model, train_docs, dev_docs, C, device, log=print, dev_sub=None, ckpt_save=None, ckpt_load=None):
    """Trains on train_docs in EPOCHS 'epochs' of EPOCH_DOCS documents each (a stream over a reshuffled permutation, so a huge training set can be seen
    partially). After every epoch a fixed sample of the selection set is scored; the best epoch's state is kept (SELECT_METRIC: f1_tuned = macro-F1 at the
    best global threshold, f1 = at 0.5). Optional model hooks (AdaLoRA): extra_loss(), on_before_step(step, total)."""
    model.to(device)
    torch.manual_seed(C["SEED"]); random.seed(C["SEED"])
    N = len(train_docs)
    per_epoch = min(C["EPOCH_DOCS"] or N, N)
    epochs = C["EPOCHS"]
    Ytr = np.stack([d["y"] for d in train_docs])
    pw = pos_weights(Ytr, C["PW_POWER"], C["PW_CAP"]).to(device) if C["LOSS"] == "bce_pw" else None
    model.head.init_bias(Ytr.mean(0))
    head_params = [p for p in model.head_parameters() if p.requires_grad]
    hp = set(id(p) for p in head_params)
    other = [p for p in model.parameters() if p.requires_grad and id(p) not in hp]
    groups = [dict(params=head_params, lr=C["HEAD_LR"])]
    if other:
        groups.append(dict(params=other, lr=C["LR"]))
    opt = torch.optim.AdamW(groups, weight_decay=C["WEIGHT_DECAY"], betas=(0.9, 0.98))
    steps_per_epoch = math.ceil(per_epoch / (C["BATCH"] * C["ACCUM"]))
    total = steps_per_epoch * epochs
    sched = torch.optim.lr_scheduler.LambdaLR(opt, _lr_lambda(total, int(C["WARMUP_FRAC"] * total)))
    use_amp = device.type == "cuda" and C["AMP"] == "fp16"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    kd_on = C.get("KD_ALPHA", 0) > 0 and all(d.get("kd") is not None for d in train_docs)
    rng = random.Random(C["SEED"] * 1000 + 7)
    stream = []
    if C.get("SAMPLING", "uniform") == "rare":      # rare-statute oversampling: a case is drawn with probability ~ max over its statutes of (1 / support) ** RARE_POWER
        sup = Ytr.sum(0).astype(np.float64); wl = (1.0 / np.maximum(sup, 1)) ** C.get("RARE_POWER", 0.5)
        wdoc = (Ytr * wl[None, :]).max(1); wdoc = (wdoc / wdoc.mean()).tolist()
    def take(k):
        nonlocal stream
        while len(stream) < k:
            if C.get("SAMPLING", "uniform") == "rare":
                stream += rng.choices(range(N), weights=wdoc, k=N)
            else:
                p = list(range(N)); rng.shuffle(p); stream += p
        out, stream = stream[:k], stream[k:]
        return out
    dev_idx = list(range(len(dev_docs)))
    if dev_sub and dev_sub < len(dev_docs):
        dev_idx = sorted(random.Random(1).sample(dev_idx, dev_sub))
    dev_eval = [dev_docs[i] for i in dev_idx]
    Yd = np.stack([d["y"] for d in dev_eval])
    hist, best, best_state, best_ep, bad = [], -1.0, None, 0, 0
    t0, tokens, gstep = time.time(), 0, 0
    start_ep = 1
    ck = ckpt_load() if ckpt_load else None
    if ck is not None:          # resume after the last finished epoch (a previous session was stopped)
        load_trainable_state(model, ck["model"])
        opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
        rng.setstate(ck["rng"]); stream = ck["stream"]; torch.set_rng_state(ck["torch_rng"])
        if device.type == "cuda" and ck.get("cuda_rng") is not None:
            torch.cuda.set_rng_state(ck["cuda_rng"], device)
        hist, best, best_state, best_ep, bad = ck["hist"], ck["best"], ck["best_state"], ck["best_ep"], ck["bad"]
        tokens, gstep = ck["tokens"], ck["gstep"]; t0 = time.time() - ck["elapsed"]; start_ep = ck["epoch"] + 1
        log(f"  resumed from the checkpoint after epoch {ck['epoch']} (best so far: epoch {best_ep}, {best:.4f})")
    log(f"training: {N} train documents, {per_epoch} per epoch x {epochs} epochs, {steps_per_epoch} optimizer steps/epoch, loss {C['LOSS']}, head {C['DOC_HEAD']}" + (", KD on" if kd_on else "")
        + (f"; pos_weight range {float(pw.min()):.2f}-{float(pw.max()):.2f}" if pw is not None else ""))
    for ep in range(start_ep, epochs + 1):
        model.train()
        order = take(per_epoch)
        run_loss, nb, skipped, pending = 0.0, 0, 0, 0
        opt.zero_grad(set_to_none=True)
        for bi in range(0, len(order), C["BATCH"]):
            b = to_device(collate(train_docs, order[bi:bi + C["BATCH"]], C), device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(b)
            logits = logits.float()
            loss = ml_loss(logits, b["labels"], C, pw)
            if kd_on:
                loss = (1 - C["KD_ALPHA"]) * loss + C["KD_ALPHA"] * kd_loss(logits, b["kd"], C["KD_TEMPERATURE"], C)
            if hasattr(model, "extra_loss"):
                loss = loss + model.extra_loss()
            if not torch.isfinite(loss):
                skipped += 1
                continue
            scaler.scale(loss / C["ACCUM"]).backward()
            nb += 1; pending += 1; tokens += int(b["mask"].sum()); run_loss += loss.item()
            if pending >= C["ACCUM"] or (bi + C["BATCH"] >= len(order) and pending > 0):
                scaler.unscale_(opt)
                if hasattr(model, "on_before_step"):
                    model.on_before_step(gstep, total)
                torch.nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], C["GRAD_CLIP"])
                scaler.step(opt); scaler.update(); sched.step(); gstep += 1
                opt.zero_grad(set_to_none=True); pending = 0
        row = dict(epoch=ep, train_loss=round(run_loss / max(1, nb), 5), train_seconds=round(time.time() - t0, 1))
        if skipped:
            row["skipped_nonfinite_batches"] = skipped
        lg, einfo = predict_logits(model, dev_eval, C, device)
        P = sigmoid_np(lg)
        s05 = ml_scores(Yd, apply_thr(P, 0.5)); thr, ft = best_global(Yd, P)
        sc_tuned = ml_scores(Yd, apply_thr(P, thr))
        score = s05["macro_F1"] if C["SELECT_METRIC"] == "f1" else ft
        row.update(dev_macroF1=round(s05["macro_F1"], 4), dev_macroF1_tuned=round(ft, 4), tuned_thr=round(thr, 2), dev_microF1_tuned=round(sc_tuned["micro_F1"], 4),
                   dev_subset_acc_tuned=round(sc_tuned["subset_acc"], 4), eval_seconds=einfo["seconds"])
        if score > best:
            best, best_ep, best_state, bad = score, ep, trainable_state(model), 0
        else:
            bad += 1
        hist.append(row)
        log("  epoch %d: " % ep + ", ".join(f"{k}={v}" for k, v in row.items() if k != "epoch"))
        if ckpt_save and ep < epochs:
            ckpt_save(dict(epoch=ep, model=trainable_state(model), opt=opt.state_dict(), sched=sched.state_dict(), scaler=scaler.state_dict(),
                           rng=rng.getstate(), stream=list(stream), torch_rng=torch.get_rng_state(),
                           cuda_rng=torch.cuda.get_rng_state(device) if device.type == "cuda" else None,
                           hist=hist, best=best, best_state=best_state, best_ep=best_ep, bad=bad, tokens=tokens, gstep=gstep, elapsed=time.time() - t0))
        if nb == 0:
            log("  every batch of this epoch was non-finite: stopping"); break
        if C["PATIENCE"] and bad >= C["PATIENCE"]:
            log(f"  early stop: no improvement for {bad} epochs"); break
        if C.get("TIME_BUDGET_MIN") and (time.time() - t0) / 60 > C["TIME_BUDGET_MIN"]:
            log(f"  time budget of {C['TIME_BUDGET_MIN']} min reached: stopping after epoch {ep}"); break
    train_secs = time.time() - t0 - sum(r["eval_seconds"] for r in hist)
    peak = torch.cuda.max_memory_allocated(device) / 1e9 if device.type == "cuda" else None
    return dict(history=hist, best_epoch=best_ep, best_score=best, best_state=best_state, train_seconds=round(train_secs, 1),
                peak_mem_gb=round(peak, 3) if peak else None, tokens_per_sec=round(tokens / max(train_secs, 1e-9), 1))


# ----------------------------------------------------------------------------------------------------------------------
# 6. Submission
# ----------------------------------------------------------------------------------------------------------------------
def write_submission(ids, pred_matrix, out_dir, name, expected_ids, n_labels):
    """JSON list [{"id": ..., "pred": [statute ids]}] (one entry per test case, sorted unique integers in 0..L-1) + a zip with prediction.json at its root."""
    assert list(ids) == list(expected_ids) or set(ids) == set(expected_ids), "ids differ from test.json"
    assert len(set(ids)) == len(ids) == len(expected_ids), "duplicate or missing ids"
    pm = np.asarray(pred_matrix).astype(bool); assert pm.shape == (len(ids), n_labels)
    sub = [dict(id=i, pred=[int(x) for x in np.flatnonzero(row)]) for i, row in zip(ids, pm)]
    assert all(len(set(s["pred"])) == len(s["pred"]) and all(isinstance(x, int) and 0 <= x < n_labels for x in s["pred"]) for s in sub)
    os.makedirs(out_dir, exist_ok=True)
    jpath = os.path.join(out_dir, f"{name}_prediction.json")
    with open(jpath, "w", encoding="utf-8") as f:
        json.dump(sub, f)
    zpath = os.path.join(out_dir, f"submission_{name}.zip")
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(jpath, arcname="prediction.json")
    return jpath, zpath


# ===================== The Phase-5 evaluator (from the course file evaluate_phase5.py; read_truth / read_pred / the three scores are unchanged, main() became a function) =====================
# Truth JSON: a list of objects, [{"id":"...", "gold":[0,2,5,...]}, ...]      Prediction JSON: [{"id":"...", "pred":[0,2,5,...]}, ...]
import argparse, json, sys
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def read_truth(path):
    try:
        x=json.load(open(path,encoding="utf-8"))
    except json.JSONDecodeError as e:
        raise ValueError(f"Truth must be a JSON file containing a list of records: {e}")
    if not isinstance(x,list) or not x: raise ValueError('Truth must be a non-empty JSON list.')
    out={}; max_label=-1
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'gold' not in r: raise ValueError(f'Truth item {n} must contain id and gold.')
        if not isinstance(r['id'],str) or not r['id'].strip() or not isinstance(r['gold'],list): raise ValueError(f'Invalid truth item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate truth ID: {r["id"]}')
        seen=set()
        for label in r['gold']:
            if isinstance(label,bool) or not isinstance(label,int) or label<0 or label in seen: raise ValueError(f'Invalid/duplicate label in truth item {n}.')
            seen.add(label); max_label=max(max_label,label)
        out[r['id']]=r['gold']
    return out,max_label+1

def read_pred(path,nlab):
    x=json.load(open(path,encoding='utf-8'))
    if not isinstance(x,list) or not x: raise ValueError('Prediction must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'pred' not in r: raise ValueError(f'Prediction item {n} must contain id and pred.')
        if not isinstance(r['id'],str) or not isinstance(r['pred'],list) or r['id'] in out: raise ValueError(f'Invalid or duplicate prediction item {n}.')
        seen=set()
        for x in r['pred']:
            if isinstance(x,bool) or not isinstance(x,int) or x<0 or x>=nlab or x in seen: raise ValueError(f'Invalid/duplicate label in prediction item {n}.')
            seen.add(x)
        out[r['id']]=r['pred']
    return out

def official_eval(truth_path, prediction_path, verbose=True):
    """Same checks and the same three numbers as evaluate_phase5.py: Macro-F1, Micro-F1, Subset Accuracy."""
    t,n=read_truth(truth_path); p=read_pred(prediction_path,n)
    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")
    yt=np.zeros((len(t),n),dtype=np.uint8); yp=np.zeros((len(t),n),dtype=np.uint8)
    for i,k in enumerate(t):
        for x in t[k]: yt[i,x]=1
        for x in p[k]: yp[i,x]=1
    res=dict(examples=len(t), labels=n, macro_F1=float(f1_score(yt,yp,average="macro",zero_division=0)),
             micro_F1=float(f1_score(yt,yp,average='micro',zero_division=0)), subset_acc=float(accuracy_score(yt,yp)))
    if verbose:
        print(f'  [official evaluator] Examples: {res["examples"]}  Labels: {res["labels"]}  Macro-F1: {res["macro_F1"]:.6f}  Micro-F1: {res["micro_F1"]:.6f}  Subset Accuracy: {res["subset_acc"]:.6f}')
    return res


def check_test_file(pred_path, test_ids, n_labels):
    """The checks Codabench applies to a submission (read_pred) + exactly one object per test case."""
    p = read_pred(pred_path, n_labels)
    if set(p) != set(test_ids) or len(p) != len(test_ids):
        raise ValueError(f"ID mismatch with test.json: missing={len(set(test_ids) - set(p))}, extra={len(set(p) - set(test_ids))}")
    return True


def dev_codabench_check(ids, Y, pred, out_dir, name, verbose=True):
    """Writes the dev truth (as {"id", "gold"}) and the predictions (as {"id", "pred"}) as files and scores them with the official evaluator code.
    Run `python evaluate_phase5.py --truth <truth> --prediction <prediction>` on the same files to get the same numbers."""
    os.makedirs(out_dir, exist_ok=True)
    assert all(isinstance(i, str) for i in ids), "case ids must be strings (the evaluator requires it)"
    Y, pred = np.asarray(Y).astype(bool), np.asarray(pred).astype(bool)
    tpath, ppath = os.path.join(out_dir, "dev_truth.json"), os.path.join(out_dir, f"{name}_dev_prediction.json")
    json.dump([dict(id=i, gold=[int(x) for x in np.flatnonzero(r)]) for i, r in zip(ids, Y)], open(tpath, "w"))
    json.dump([dict(id=i, pred=[int(x) for x in np.flatnonzero(r)]) for i, r in zip(ids, pred)], open(ppath, "w"))
    return official_eval(tpath, ppath, verbose), tpath, ppath

In [3]:
import numpy as np
SWEEP_SUFFIX = ""                       # (testing only)
MEMBERS = [('p5_moe', 'moe_partial2'), ('p5_gqa', 'gqa_lorma'), ('p5_gqa', 'gqa_partial4'), ('p5_gdn', 'gdn_lorma'), ('p5_lstm', 'lstm_full_ft'), ('p5_lstm', 'lstm_frozen'), ('p5_kd', 'kd_gqa_lorma'), ('p5_kd', 'kd_gqa_partial4'), ('p5_kd_lstm', 'kd_lstm_full_ft'), ('p5_mor', 'mor_partial4'), ('p5_moe_long', 'moe_partial2_long'), ('p5_moe_lorma', 'moe_lorma_long'), ('p5_gqa_long', 'gqa_partial4_long'), ('p5_gqa_long', 'gqa_lorma_long'), ('p5_gqa_long', 'gqa_lora_long'), ('p5_gdn2', 'gdn_partial4'), ('p5_gdn2', 'gdn_full_ft'), ('p5_mor2', 'mor_lorma'), ('p5_mor2', 'mor_full_ft'), ('p5_kd_long', 'kd_gqa_lorma_long'), ('p5_kd_long', 'kd_gqa_partial4_long'), ('p5_kd_lstm', 'kd_lstm_full_ft'), ('p5_lstm_long', 'kd_lstm_full_ft_long'), ('p5_mor_long', 'mor_full_ft_long'), ('p5_mor_long', 'kd_mor_full_ft_long'), ('p5_moe_attn', 'moe_attn_long'), ('p5_moe_attn_rare', 'moe_attn_rare_long'), ('p5_kd_attn', 'kd_gqa_partial4_attn_long'), ('p5_kd_attn', 'kd_gqa_lorma_attn_long')]                     # (notebook, setup) pairs of runs trained on ALL training cases; missing ones are skipped
MIN_GAIN, MAX_MEMBERS = 0.002, 6        # greedy selection: a member is added only if the selection-set macro-F1 (best global threshold) improves by more than MIN_GAIN
POOL_MIN_F1 = 0.0                       # the "all good" ensemble uses the members whose own selection-set F1 is at least this (0.0 = overwritten below by 97% of the best member)
HONEST_REPS = 8
R = {}
for sweep, setup in MEMBERS:
    name = f"p5res_{sweep}{SWEEP_SUFFIX}__{setup}.pt"
    try:
        R[setup] = torch.load(fetch_model_file(name), map_location="cpu", weights_only=False)
    except Exception as e:
        print(f"  (skipped {name}: {type(e).__name__})")
names = sorted(R)
assert len(names) >= 2, "fewer than two cached results found"
TEST_IDS = R[names[0]]["test_ids"]
for n in names:
    assert R[n]["test_ids"] == TEST_IDS, f"{n}: different test cases"
common = sorted(set.intersection(*[set(i for i, k in zip(R[n]["dev_ids"], R[n]["dev_sel_mask"]) if k) for n in names]))
pos = {n: {i: k for k, i in enumerate(R[n]["dev_ids"])} for n in names}
n0 = names[0]
Y = np.stack([R[n0]["dev_y"][pos[n0][i]] for i in common]); NL = Y.shape[1]
print(f"{len(names)} members; selection-set cases common to all members: {len(common)}; test cases {len(TEST_IDS)}; statutes {NL}")
for n in names:
    assert all((R[n]["dev_y"][pos[n][i]] == Y[k]).all() for k, i in enumerate(common[:200])), f"{n}: labels differ"
lg = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / np.clip(1 - p, 1e-6, 1))
# Members are calibrated differently (positive weights, KD ...): each member's score is shifted so that its own best thresholds sit at 0: s = logit(p) - logit(thr_l).
# The ensemble averages these shifted logits; its thresholds are then fitted again on the ensemble probabilities.
SD, ST, OWN = {}, {}, {}
for n in names:
    r = R[n]; mode = r["scores"]["best_mode"]; thr = np.asarray(r["scores"]["thr"][mode])
    pdv = sigmoid_np(r["dev_logits"][[pos[n][i] for i in common]])
    SD[n] = lg(pdv) - lg(thr); ST[n] = lg(sigmoid_np(r["test_logits"])) - lg(thr)
    OWN[n] = best_global(Y, sigmoid_np(SD[n]))[1]
ranked = sorted(names, key=lambda n: -OWN[n])
POOL_MIN_F1 = 0.97 * OWN[ranked[0]]
print(f"{'member':24s} {'macro-F1 (best global thr, shifted)':>36s}")
for n in ranked:
    print(f"{n:24s} {OWN[n]:36.4f}")

def ens_prob(mem, which=SD, idx=slice(None)):
    return sigmoid_np(np.mean([which[m][idx] for m in mem], 0))
def tuned(mem, idx=slice(None)):
    return best_global(Y[idx], ens_prob(mem, SD, idx))[1]
def greedy(idx):
    chosen, best = [], -1.0
    while len(chosen) < MAX_MEMBERS:
        cand = [(tuned(chosen + [n], idx), n) for n in names if n not in chosen]
        if not cand: break
        s, n = max(cand)
        if chosen and s - best <= MIN_GAIN: break
        chosen.append(n); best = s
    return chosen
RULES = {"greedy": greedy,
         "top3": lambda idx: sorted(names, key=lambda n: -tuned([n], idx))[:3],
         "top5": lambda idx: sorted(names, key=lambda n: -tuned([n], idx))[:5],
         "all_good": lambda idx: [n for n in names if tuned([n], idx) >= POOL_MIN_F1] or ranked[:1],
         "best_single": lambda idx: [max(names, key=lambda n: tuned([n], idx))]}

# honest check: choose members AND thresholds on one random half of the selection set, score on the other half
print("\nhonest check (members and thresholds chosen on one half of the selection set, macro-F1 on the other half; mean over splits):")
HONEST = {}
for rule, f in RULES.items():
    sc = []
    for seed in range(HONEST_REPS):
        perm = np.random.RandomState(seed).permutation(len(Y)); A, B = perm[:len(Y) // 2], perm[len(Y) // 2:]
        mem = f(A); P = ens_prob(mem)
        thr = fit_thresholds(Y[A], P[A], "shrunk")
        sc.append(ml_scores(Y[B], apply_thr(P[B], thr))["macro_F1"])
    HONEST[rule] = float(np.mean(sc))
    print(f"  {rule:12s} {np.mean(sc):.4f} (sd {np.std(sc):.4f})")

SUB_DIR = f"{WORK}/p5_submissions_ensemble"
os.makedirs(SUB_DIR, exist_ok=True)
FINAL = {}
print("\nfinal ensembles (members and thresholds chosen on the whole selection set: optimistic):")
for rule, f in RULES.items():
    mem = f(slice(None)); sd_, st_ = ens_prob(mem), ens_prob(mem, ST)
    hon, _ = cross_fit(Y, sd_, reps=4); mode = max(hon, key=hon.get)
    thr = fit_thresholds(Y, sd_, mode); s = ml_scores(Y, apply_thr(sd_, thr))
    FINAL[rule] = dict(members=mem, mode=mode, sel_f1=s["macro_F1"], micro_F1=s["micro_F1"], subset_acc=s["subset_acc"], honest_thr_only=hon[mode], honest_members_and_thr=HONEST[rule])
    preds = apply_thr(st_, thr)
    _, zpath = write_submission(TEST_IDS, preds, SUB_DIR, f"p5_ensemble_{rule}", TEST_IDS, NL)
    upload_model_file(zpath)
    check_test_file(os.path.join(SUB_DIR, f"p5_ensemble_{rule}_prediction.json"), TEST_IDS, NL)
    try:
        dev_codabench_check(common, Y, apply_thr(sd_, thr), SUB_DIR, f"p5_ensemble_{rule}")
    except Exception as ex:
        print(f"  (official evaluator check on dev skipped: {type(ex).__name__}: {ex})")
    print(f"  {rule:12s} sel macro-F1 {s['macro_F1']:.4f} micro {s['micro_F1']:.4f} subset-acc {s['subset_acc']:.4f} ({mode} thr) | honest {HONEST[rule]:.4f} | {preds.sum(1).mean():.2f} statutes/case on test (dev gold {Y.sum(1).mean():.2f}) | {os.path.basename(zpath)}\n      members: {mem}")
json.dump(dict(members=names, own_f1=OWN, final=FINAL), open(f"{WORK}/summary_p5_ensemble.json", "w"), indent=1, default=float)
upload_model_file(f"{WORK}/summary_p5_ensemble.json")
print("\nsubmit the zips in order of the honest score (it is the fair estimate); the sel F1 of greedy/top-k is chosen on the same cases and is optimistic.")

28 members; selection-set cases common to all members: 10079; test cases 13019; statutes 100
member                    macro-F1 (best global thr, shifted)
moe_attn_long                                          0.3909
moe_partial2_long                                      0.3837
kd_gqa_partial4_attn_long                               0.3836
kd_gqa_partial4_long                                   0.3822
moe_lorma_long                                         0.3777
moe_partial2                                           0.3730
gqa_partial4_long                                      0.3705
kd_gqa_lorma_long                                      0.3658
gqa_lora_long                                          0.3594
kd_gqa_lorma_attn_long                                 0.3580
gqa_lorma_long                                         0.3569
gqa_partial4                                           0.3556
kd_gqa_partial4                                        0.3508
kd_gqa_lorma                          

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded submission_p5_ensemble_greedy.zip to bucket kavincg23/cs781-models
  [official evaluator] Examples: 10079  Labels: 100  Macro-F1: 0.402126  Micro-F1: 0.503184  Subset Accuracy: 0.053676
  greedy       sel macro-F1 0.4021 micro 0.5032 subset-acc 0.0537 (global thr) | honest 0.3986 | 4.26 statutes/case on test (dev gold 3.91) | submission_p5_ensemble_greedy.zip
      members: ['moe_attn_long', 'kd_gqa_partial4_long']


Processing Files (0 / 0): |          |  0.00B /  0.00B            

... (15 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded summary_p5_ensemble.json to bucket kavincg23/cs781-models

submit the zips in order of the honest score (it is the fair estimate); the sel F1 of greedy/top-k is chosen on the same cases and is optimistic.
